## Decision Tree from Stratch

### Libraries

In [103]:
import pandas as pd
import numpy as np

# interpretation of result
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# reference implementation


### Load Case 1

In [104]:
df_train1 = pd.read_csv('../data/case1/train1.csv')
df_test1 = pd.read_csv('../data/case1/test1.csv')

### Change relevant columns

In [105]:
# speed in krpm, n
df_train1['krpm'] = df_train1['speed_rpm'] / 1000

# pressure rise across pump
df_train1['delta_p'] = (
    df_train1['outlet_pressure_bar'] - df_train1['inlet_pressure_bar']
)

# speed in krpm, n
df_test1['krpm'] = df_test1['speed_rpm'] / 1000

# pressure rise across pump
df_test1['delta_p'] = (
    df_test1['outlet_pressure_bar'] - df_test1['inlet_pressure_bar']
)


In [106]:
exclude_cols = [
    'pump_id',
    'elapsed_s',
    'operating_state',
    'inlet_pressure_bar',
    'outlet_pressure_bar',
    'flow_m3_h',
    'speed_rpm'
]

X_train = df_train1.drop(columns = exclude_cols)
y_train = df_train1['flow_m3_h']

X_test = df_test1.drop(columns = exclude_cols)
y_test = df_test1['flow_m3_h']

In [107]:
X_train

,power_w,krpm,delta_p
0,669.992714,4.230396,1.357291
1,931.528480,3.796520,0.293043
2,724.198083,3.508264,0.185766
3,99.463217,1.498405,0.163881
4,608.195308,3.224874,0.231682
...,...,...,...
475,815.229829,4.376002,1.462426
476,515.961500,3.514587,0.927272
477,534.173942,2.935130,0.499988
478,154.677482,2.214153,0.378757


In [108]:
X_train = np.array(X_train)
y_train = np.array(y_train)

X_test = np.array(X_test)
y_test = np.array(y_test)

### Create Helper Functions

In [109]:
class RelU:
    def forward(inputs):
        return np.maximum(0, inputs, dtype=np.float64)

    def backward(inputs):
        return np.array(inputs > 0, dtype=np.float64)

class Cost:
    def cost_forward(ypred, ytrue):
        return 1 / 2 * (np.sum(np.square(ytrue - ypred)))
        
    def cost_backward(ypred, ytrue):
        return ypred - ytrue

class SGD:
    def __init__(self, learning_rate=0.005):
        self.lr = learning_rate

    def generate_nabla(self, delta, prev_activation):
        m = delta.shape[1]  
        nabla_w = self.lr * (delta @ prev_activation.T) / m
        nabla_b = self.lr * delta.mean(axis = 1, keepdims = True)
        return nabla_w, nabla_b

In [110]:
class FNN:
    def __init__(self, units: int, activation=None, input_dim: int = False):
        self.units = units
        self.activation = activation
        self.input_dim = input_dim

        self.weights = None
        self.bias = None

    def initialize(self, prev_units: int, mean: float = 0.0, stdev: float = 0.1):
        self.weights = np.random.normal(
            loc=mean, scale=stdev, size=(self.units, prev_units)
        )
        self.bias = np.ones((self.units, 1))

    def forward(self, inputs):
        z = self.weights @ inputs + self.bias
        if self.activation is not None:
            return z, self.activation.forward(z)
        return z, z

    def backward(self, propagated_error, current_z): 
        ''' 
        Returns:
            delta (float): change in cost if the value at this layer changes
            next_propagted_error (float): change in cost if previous layer changed
        '''
        delta = propagated_error
        if self.activation is not None:
            delta = delta * self.activation.backward(current_z)
        next_propagated_error = self.weights.T @ delta
        return delta, next_propagated_error

    def update(self, nabla_w, nabla_b):
        self.weights = self.weights - nabla_w
        self.bias = self.bias - nabla_b

In [111]:
import numpy as np
import copy

class Sequential:
    def __init__(self, layers = None):
        self.layers = layers if layers is not None else []

    def add(self, layer):
        self.layers.append(layer)

    def _initialize_weights_and_bias(self, input_shape):
        for layer in self.layers:
            layer.initialize(input_shape)
            input_shape = layer.units

    def compile(self, optimizer=None, cost=Cost):
        self.optimizer = optimizer if optimizer is not None else SGD()
        self.cost = cost
    
    def evaluate(self, X, y):
        y = y.reshape(-1)
        ypred = self.predict(X)
        cost = self.cost.cost_forward(ypred, y)

        return ypred , cost

    def predict(self, X):
        _, a = self.layers[0].forward(X.T)
        for j in range(1, len(self.layers)):
            _, a = self.layers[j].forward(a)
        return a

    def fit(
        self, X, y, batch_size=1, epochs=1, verbose=1, history=False, save_best=False
    ):
        y = y.reshape(-1)
        if not self.layers[0].input_dim:
            raise ValueError("Please specify input_dim on the first layer")

        total_samples = len(y)
        samples_per_batch = int(total_samples / batch_size)

        number_of_layers = len(self.layers)
        if self.layers[0].weights is None:  self._initialize_weights_and_bias(input_shape=self.layers[0].input_dim)

        costs = []
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).reshape(-1)
        total_samples = len(y)

        best_layers = None
        best_cost = np.inf

        for i in range(epochs):
            idx = np.random.permutation(total_samples)
            for start in range(0, total_samples, batch_size):
                b = idx[start:start + batch_size]
                X_taken, y_taken = X[b].T, y[b]

                # Forward pass
                z, a = self.layers[0].forward(X_taken)
                zs, activations = [z], [a]
                for j in range(1, number_of_layers):
                    z, a = self.layers[j].forward(activations[-1])
                    zs.append(z)
                    activations.append(a)

                # Backward pass
                propagated_error = self.cost.cost_backward(activations[-1], y_taken)
                for j in range(number_of_layers - 1, -1, -1):
                    delta, propagated_error = self.layers[j].backward(
                        propagated_error, zs[j]
                    )

                    prev_activation = X_taken if j == 0 else activations[j - 1]

                    nabla_w, nabla_b = self.optimizer.generate_nabla(delta, prev_activation)
                    self.layers[j].update(nabla_w, nabla_b)
                    
             # -------------------------------
            cost = self.cost.cost_forward(self.predict(X), y) / total_samples   # mean cost

            if cost < best_cost:
                best_cost = cost
                best_layers = copy.deepcopy(self.layers)

            costs.append(cost)

            if verbose == 1:
                print(f"Epoch ({i + 1}/{epochs}), cost = {cost}")
            # -------------------------------
        
        if save_best and best_layers is not None:            
            self.layers = best_layers
            print("Best cost:", best_cost)

        return costs

In [112]:
new_X_train, X_val, new_y_train, y_val = train_test_split(
    X_train, y_train, test_size = 0.2, random_state = 42
)

x_scaler = StandardScaler()
new_X_train = x_scaler.fit_transform(new_X_train)
X_val = x_scaler.transform(X_val)
X_test = x_scaler.transform(X_test)

y_scaler = StandardScaler()
new_y_train = y_scaler.fit_transform(new_y_train.reshape(-1 , 1))
y_val = y_scaler.transform(y_val.reshape(-1 , 1))
y_test = y_scaler.transform(y_test.reshape(-1 , 1))

In [120]:
model = Sequential()
model.add(FNN(32, RelU, input_dim=new_X_train.shape[1]))
model.add(FNN(16, RelU))
model.add(FNN(1))
model.compile(optimizer=SGD(0.04), cost=Cost)
costs = model.fit(new_X_train, new_y_train, batch_size=32, epochs=100, verbose=0)

def score(X_s, y_s):
    pred = y_scaler.inverse_transform(model.predict(X_s).reshape(-1, 1)).ravel()
    true = y_scaler.inverse_transform(np.asarray(y_s).reshape(-1, 1)).ravel()
    return np.sqrt(mean_squared_error(true, pred)), r2_score(true, pred)

print("train", score(new_X_train, new_y_train))
print("val  ", score(X_val, y_val))
print("test ", score(X_test, y_test))

train (np.float64(1.567412607643373), 0.9859336635038584)
val   (np.float64(1.725048658219131), 0.9801604802934409)
test  (np.float64(1.4900359865667825), 0.9847667105899796)
